# Evaluation Pipeline — All 7 LoRA Models (A100 Optimized)

**Metrics:** LPIPS ↓ · SSIM ↑ · PSNR ↑ · Time Acc. ↑ · Weather Acc. ↑  
**Modes:** `best_of=1` (Table 1) · `best_of=4` (Table 2)  
**Visuals:** Best-of-test grids per model — source → controls → generated → reference

A100 optimizations used:
- `bfloat16` (A100 native tensor core dtype)
- TF32 + cuDNN benchmark
- Flash Attention 2
- `torch.compile(mode='reduce-overhead')` on the UNet
- Control maps pre-cached for all 42 samples before inference begins
- Parallel control map generation (ThreadPoolExecutor)
- Batched CLIP condition detection

## Step 1: Install Dependencies

In [ ]:
# Core stack pinned to versions used during training
!pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers[torch]==0.31.0 transformers==4.44.2 accelerate==0.34.2 peft==0.13.2

# Flash Attention 2 — large speed-up on A100 for attention layers
!pip install -q flash-attn --no-build-isolation

# Metrics + image utils
!pip install -q lpips scikit-image opencv-python tqdm pandas openpyxl

print('✓ All dependencies installed')

## Step 2: Imports & A100 Setup

In [ ]:
import os, gc, math, warnings, re
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn.functional as F
import cv2
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from skimage.metrics import structural_similarity as skssim
from skimage.metrics import peak_signal_noise_ratio as skpsnr
import lpips as lpips_module

from diffusers import (
    # Img2Img variant — initialises from source image instead of pure noise.
    # Critical for style transfer: preserves building structure while changing
    # lighting/atmosphere. Previously used txt2img which regenerated from scratch.
    StableDiffusionControlNetImg2ImgPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
    UNet2DConditionModel,
)
from transformers import (
    CLIPModel, CLIPProcessor,
    DPTForDepthEstimation, DPTImageProcessor,
    OneFormerProcessor, OneFormerForUniversalSegmentation,
)
from peft import PeftModel

warnings.filterwarnings('ignore')

# ── A100-specific global flags ────────────────────────────────────────────────
assert torch.cuda.is_available(), 'Switch runtime to GPU (A100 preferred)'
device = 'cuda'
dtype  = torch.bfloat16        # A100 native bf16 tensor cores; more stable than f16

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True
torch.set_float32_matmul_precision('high')

print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'dtype: {dtype}  |  TF32: enabled  |  cudnn.benchmark: enabled')

## Step 3: Mount Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── UPDATE IF YOUR DRIVE LAYOUT DIFFERS ─────────────────────────────────────
BASE        = '/content/drive/MyDrive/5190 S26 Final Project'
VAL_DIR     = f'{BASE}/Dataset/ValidationSet'   # 21 numbered sub-folders, 2 images each
TRAIN_OUT   = f'{BASE}/Training_Outputs'         # {Name}/lora_checkpoints/lora_best
RESULTS_DIR = f'{BASE}/Evaluation_Results'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/visuals', exist_ok=True)

SD_BASE = 'runwayml/stable-diffusion-v1-5'

CONFIGS = {
    1: {'name': 'Canny',       'controls': ['canny'],                  'cn_ids': ['lllyasviel/control_v11p_sd15_canny']},
    2: {'name': 'Segmentation','controls': ['seg'],                    'cn_ids': ['lllyasviel/control_v11p_sd15_seg']},
    3: {'name': 'Depth',       'controls': ['depth'],                  'cn_ids': ['lllyasviel/control_v11f1p_sd15_depth']},
    4: {'name': 'Canny+Seg',   'controls': ['canny', 'seg'],           'cn_ids': ['lllyasviel/control_v11p_sd15_canny', 'lllyasviel/control_v11p_sd15_seg']},
    5: {'name': 'Canny+Depth', 'controls': ['canny', 'depth'],         'cn_ids': ['lllyasviel/control_v11p_sd15_canny', 'lllyasviel/control_v11f1p_sd15_depth']},
    6: {'name': 'Seg+Depth',   'controls': ['seg',   'depth'],         'cn_ids': ['lllyasviel/control_v11p_sd15_seg',   'lllyasviel/control_v11f1p_sd15_depth']},
    7: {'name': 'Triple',      'controls': ['canny', 'seg', 'depth'],  'cn_ids': ['lllyasviel/control_v11p_sd15_canny', 'lllyasviel/control_v11p_sd15_seg', 'lllyasviel/control_v11f1p_sd15_depth']},
}

RESOLUTION = 512
NUM_STEPS  = 30

# ── Img2Img strength ─────────────────────────────────────────────────────────
# Controls how much the source image is preserved vs redrawn.
# 0.0 = identical to source, 1.0 = pure noise (same as txt2img).
# For weather/time transfer: 0.55 (subtle) → 0.75 (day↔night).
# Adaptive strength is computed per sample based on condition distance.
STRENGTH_WEATHER_ONLY = 0.50   # same time, different weather (sunny→cloudy)
STRENGTH_TIME_ONLY    = 0.72   # same weather, different time (day→night)
STRENGTH_BOTH         = 0.65   # different time + different weather

# ── Inference hyperparameters ─────────────────────────────────────────────────
# CFG 5.5 instead of 7.5: img2img already anchors to the source, so you need
# less prompt-push; higher CFG with img2img causes over-saturation.
CFG_SCALE  = 5.5
CN_SCALE   = 0.85
BEST_OF_1  = 1
BEST_OF_4  = 4
N_BEST_VIZ = 6

print('✓ Paths configured')
print(f'  Val  : {VAL_DIR}')
print(f'  Outs : {RESULTS_DIR}')
print(f'  Img2Img strengths — weather-only: {STRENGTH_WEATHER_ONLY} | '
      f'time-only: {STRENGTH_TIME_ONLY} | both: {STRENGTH_BOTH}')
print(f'  CFG: {CFG_SCALE} (reduced from 7.5 — img2img needs less prompt-push)')

## Step 4: Load Shared Utility Models (once)

In [ ]:
# ── Depth (DPT-Large) ─────────────────────────────────────────────────────────
print('Loading DPT depth model...')
depth_model     = DPTForDepthEstimation.from_pretrained('Intel/dpt-large', torch_dtype=dtype).to(device)
depth_processor = DPTImageProcessor.from_pretrained('Intel/dpt-large')
depth_model.eval()

# ── Segmentation (OneFormer ADE20k-Swin-Tiny) ────────────────────────────────
print('Loading OneFormer segmentation model...')
seg_processor = OneFormerProcessor.from_pretrained('shi-labs/oneformer_ade20k_swin_tiny')
seg_model     = OneFormerForUniversalSegmentation.from_pretrained(
    'shi-labs/oneformer_ade20k_swin_tiny', torch_dtype=dtype
).to(device)
seg_model.eval()

# ── LPIPS (AlexNet backbone — fastest + well-correlated with human perception) ─
print('Loading LPIPS...')
lpips_fn = lpips_module.LPIPS(net='alex').to(device)
lpips_fn.eval()

# ── CLIP (ViT-B/32 — condition accuracy) ─────────────────────────────────────
print('Loading CLIP...')
clip_model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()

torch.cuda.empty_cache(); gc.collect()
print('\n✓ All shared models loaded')

## Step 5: Helper Functions (Control Maps · Metrics · CLIP)

In [ ]:
# ── Control map generators ─────────────────────────────────────────────────────

def make_canny(image: Image.Image) -> Image.Image:
    gray = np.array(image.convert('L'))
    bright = np.mean(gray)
    low, high, k = (35, 90, (5, 5)) if bright < 70 else (80, 160, (3, 3))
    edges = cv2.Canny(cv2.GaussianBlur(gray, k, 0), low, high)
    return Image.fromarray(cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB))


@torch.no_grad()
def make_depth(image: Image.Image) -> Image.Image:
    inputs = {k: v.to(device, dtype=dtype)
              for k, v in depth_processor(images=image, return_tensors='pt').items()}
    depth = depth_model(**inputs).predicted_depth
    depth = F.interpolate(depth.unsqueeze(1), (RESOLUTION, RESOLUTION),
                          mode='bicubic', align_corners=False).squeeze()
    arr = depth.cpu().float().numpy()
    arr = ((arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * 255).astype(np.uint8)
    return Image.fromarray(arr).convert('RGB')


@torch.no_grad()
def make_seg(image: Image.Image) -> Image.Image:
    img_r = image.resize((RESOLUTION, RESOLUTION), Image.LANCZOS)
    inputs = seg_processor(images=img_r, task_inputs=['semantic'], return_tensors='pt')
    inputs = {k: v.to(device, dtype=dtype) if isinstance(v, torch.Tensor) else v
              for k, v in inputs.items()}
    out  = seg_model(**inputs)
    smap = seg_processor.post_process_semantic_segmentation(
        out, target_sizes=[(RESOLUTION, RESOLUTION)]
    )[0].cpu().numpy().astype(np.uint8)
    if smap.max() > 0:
        smap = (smap / smap.max() * 255).astype(np.uint8)
    return Image.fromarray(smap).convert('RGB')


CTRL_FN = {'canny': make_canny, 'depth': make_depth, 'seg': make_seg}


# ── Metrics ────────────────────────────────────────────────────────────────────

def _to_arr(img: Image.Image):
    return np.array(img.convert('RGB').resize((RESOLUTION, RESOLUTION)))

def compute_lpips(gen: Image.Image, ref: Image.Image) -> float:
    def _t(img):
        return torch.from_numpy(
            _to_arr(img).astype(np.float32) / 127.5 - 1
        ).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        return float(lpips_fn(_t(gen), _t(ref)).item())

def compute_ssim(gen: Image.Image, ref: Image.Image) -> float:
    return float(skssim(_to_arr(gen), _to_arr(ref), channel_axis=2, data_range=255))

def compute_psnr(gen: Image.Image, ref: Image.Image) -> float:
    return float(skpsnr(_to_arr(ref).astype(np.float32),
                        _to_arr(gen).astype(np.float32), data_range=255))


# ── CLIP condition detection (batched) ─────────────────────────────────────────

TIME_PROMPTS    = ['a daytime photo', 'a nighttime photo']
WEATHER_PROMPTS = ['a sunny clear weather photo', 'a cloudy overcast photo', 'a rainy foggy photo']

@torch.no_grad()
def clip_classify_batch(images: list, prompts: list) -> list:
    inputs = clip_processor(text=prompts, images=images,
                            return_tensors='pt', padding=True).to(device)
    logits = clip_model(**inputs).logits_per_image
    return logits.softmax(dim=1).argmax(dim=1).cpu().tolist()

def detect_condition_batch(images: list) -> list:
    times    = clip_classify_batch(images, TIME_PROMPTS)
    weathers = clip_classify_batch(images, WEATHER_PROMPTS)
    return [{'time': t, 'weather': w} for t, w in zip(times, weathers)]


# ── Filename-based condition parsing (more reliable than CLIP when available) ──

_TIME_KWS    = {'night': 1, 'nighttime': 1, 'evening': 1, 'dusk': 1, 'sunset': 1,
                'dawn': 1, 'day': 0, 'daytime': 0, 'morning': 0, 'noon': 0, 'afternoon': 0}
_WEATHER_KWS = {'sunny': 0, 'clear': 0, 'sun': 0,
                'cloudy': 1, 'overcast': 1, 'cloud': 1, 'hazy': 1,
                'rain': 2, 'rainy': 2, 'fog': 2, 'foggy': 2, 'storm': 2}

def parse_condition_from_filename(path: str) -> dict | None:
    """Try to extract time/weather from Penn dataset filename conventions.
    Returns None if filename gives no signal (fall back to CLIP)."""
    stem  = os.path.splitext(os.path.basename(path))[0].lower()
    parts = re.split(r'[_\-\s]+', stem)
    time_i, weather_i = None, None
    for p in parts:
        if time_i    is None and p in _TIME_KWS:    time_i    = _TIME_KWS[p]
        if weather_i is None and p in _WEATHER_KWS: weather_i = _WEATHER_KWS[p]
    if time_i is None or weather_i is None:
        return None
    return {'time': time_i, 'weather': weather_i}


def get_condition(path: str, image: Image.Image) -> dict:
    """Filename parse first; CLIP fallback."""
    cond = parse_condition_from_filename(path)
    if cond is not None:
        return cond
    return detect_condition_batch([image])[0]


# ── Adaptive img2img strength ──────────────────────────────────────────────────

def adaptive_strength(src_cond: dict, tgt_cond: dict) -> float:
    """
    Return img2img strength based on how different the conditions are.
    Day↔Night needs more redrawing (higher strength) than just weather changes.
    """
    diff_time    = src_cond['time']    != tgt_cond['time']
    diff_weather = src_cond['weather'] != tgt_cond['weather']
    if diff_time and diff_weather: return STRENGTH_BOTH
    if diff_time:                  return STRENGTH_TIME_ONLY
    return STRENGTH_WEATHER_ONLY


# ── Detailed prompt builder ────────────────────────────────────────────────────
# Each template is specific to the target condition AND lists what to avoid
# (the source condition appears in the negative prompt for cleaner transfer).

_TIME_LABELS    = ['daytime', 'nighttime']
_WEATHER_LABELS = ['sunny', 'cloudy', 'rainy']

_POS = {
    # (time, weather) → positive prompt suffix
    (0, 0): ('bright midday sunshine, crisp hard shadows, vivid blue sky with a few clouds, '
             'golden warm sunlight on brick and stone facades, clear visibility'),
    (0, 1): ('overcast grey sky, diffuse soft shadowless lighting, flat even illumination, '
             'muted desaturated tones, thick cloud cover, no harsh shadows'),
    (0, 2): ('heavy rain, wet reflective pavements, puddles, dark overcast sky, '
             'rain streaks visible, glistening stone and brick, foggy atmosphere'),
    (1, 0): ('nighttime, clear dark sky, stars faintly visible, warm sodium street-lamp glow, '
             'windows lit from inside, deep shadows, ambient artificial light'),
    (1, 1): ('nighttime, thick overcast cloud cover, dim diffuse ambient light, '
             'street lamps glowing softly through the murk, no stars visible, moody'),
    (1, 2): ('nighttime heavy rain, wet reflective ground glowing from street lamps, '
             'dark stormy sky, rain streaks lit by artificial light, atmospheric'),
}

_NEG_TIME = {
    0: 'nighttime, dark, artificial lighting, street lamps, windows lit at night',
    1: 'daytime, sunlight, blue sky, bright sunshine, midday',
}
_NEG_WEATHER = {
    0: 'rain, fog, overcast, cloudy, grey sky, puddles',
    1: 'sunny, sunshine, clear sky, harsh shadows, bright blue sky',
    2: 'sunny, clear sky, dry ground, harsh shadows',
}

def build_prompt(src_cond: dict, tgt_cond: dict, building_hint: str = '') -> tuple:
    """
    Returns (positive_prompt, negative_prompt) pair.
    building_hint is an optional building name parsed from filename.
    """
    ti, wi = tgt_cond['time'], tgt_cond['weather']
    si, sw = src_cond['time'], src_cond['weather']

    building = (f'University of Pennsylvania campus, {building_hint}, '
                if building_hint else 'University of Pennsylvania campus building, ')

    pos = (
        f'{building}'
        f'{_TIME_LABELS[ti]}, {_WEATHER_LABELS[wi]}, '
        f'{_POS[(ti, wi)]}, '
        f'photorealistic architectural photography, ultra detailed, 8K, sharp focus, '
        f'high dynamic range, Penn campus'
    )

    neg = (
        f'blurry, low quality, distorted, watermark, text, logo, cartoon, painting, '
        f'oversaturated, deformed, ugly, disfigured, '
        f'{_NEG_TIME[si]}, {_NEG_WEATHER[sw]}'
    )

    return pos, neg


def extract_building(path: str) -> str:
    """Guess building name from filename prefix (best-effort)."""
    stem  = os.path.splitext(os.path.basename(path))[0].lower()
    parts = re.split(r'[_\-\s]+', stem)
    skip  = set(_TIME_KWS) | set(_WEATHER_KWS) | {'front','back','side','left','right',
                                                    'view','facing','sideview','photo'}
    name_parts = [p for p in parts if p not in skip and not p.isdigit()][:3]
    return ' '.join(name_parts).replace('_', ' ').title()


print('✓ Helper functions defined (Img2Img prompts + adaptive strength + filename parsing)')

## Step 6: Pipeline Loading + A100-Optimized Inference

In [ ]:
def load_pipeline(config: dict) -> StableDiffusionControlNetImg2ImgPipeline:
    """
    Load SD1.5 + ControlNet(s) + LoRA using the Img2Img pipeline.
    The Img2Img variant initialises denoising from the (partially noised) source
    image rather than pure Gaussian noise, which is essential for style transfer:
    it keeps the building geometry/materials intact and only changes the
    lighting/atmosphere guided by the prompt and strength parameter.
    """
    lora_path = f"{TRAIN_OUT}/{config['name']}/lora_checkpoints/lora_best"
    assert os.path.exists(lora_path), f'LoRA checkpoint not found:\n  {lora_path}'

    # ── ControlNets ────────────────────────────────────────────────────────────
    print(f"  Loading {len(config['cn_ids'])} ControlNet(s)...")
    cns    = [ControlNetModel.from_pretrained(m, torch_dtype=dtype).to(device)
              for m in config['cn_ids']]
    cn_arg = cns[0] if len(cns) == 1 else cns

    # ── UNet + LoRA ────────────────────────────────────────────────────────────
    print('  Loading UNet + LoRA...')
    unet = UNet2DConditionModel.from_pretrained(SD_BASE, subfolder='unet', torch_dtype=dtype)
    unet = PeftModel.from_pretrained(unet, lora_path, local_files_only=True).to(device)
    unet.eval()

    # ── Img2Img pipeline (key change from previous txt2img version) ────────────
    pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
        SD_BASE,
        unet=unet,
        controlnet=cn_arg,
        torch_dtype=dtype,
        safety_checker=None,
    ).to(device)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

    # ── A100 attention optimizations ───────────────────────────────────────────
    try:
        pipe.enable_xformers_memory_efficient_attention()
        print('  ✓ xformers enabled')
    except Exception:
        pass
    try:
        pipe.unet.enable_flash_attention()
        print('  ✓ Flash Attention 2 enabled')
    except Exception:
        pass

    print('  Compiling UNet with torch.compile...')
    pipe.unet = torch.compile(pipe.unet, mode='reduce-overhead', fullgraph=False)

    return pipe


def run_inference(
    pipe: StableDiffusionControlNetImg2ImgPipeline,
    source: Image.Image,
    ctrl_imgs: list,
    controls: list,
    pos_prompt: str,
    neg_prompt: str,
    strength: float,
    best_of: int = 1,
    ref: Image.Image = None,
) -> tuple:  # (best_image, best_lpips_or_None)
    """
    Img2Img inference with ControlNet conditioning.

    `source`   — the input building photo (acts as structural anchor)
    `strength` — how much to deviate from source (0=identical, 1=full redraw)
    `best_of`  — run N times, return candidate with lowest LPIPS vs ref
    """
    cn_scale = [CN_SCALE] * len(controls)
    img_arg  = ctrl_imgs[0] if len(ctrl_imgs) == 1 else ctrl_imgs
    sc_arg   = cn_scale[0]  if len(cn_scale)  == 1 else cn_scale

    best_img, best_lp = None, float('inf')

    with torch.inference_mode():
        for seed in range(best_of):
            generator = torch.Generator(device=device).manual_seed(seed * 1337 + 42)
            out = pipe(
                prompt=pos_prompt,
                negative_prompt=neg_prompt,
                image=source,                # ← source image as the starting point
                control_image=img_arg,       # ← control maps for structural guidance
                strength=strength,           # ← adaptive per-sample
                num_inference_steps=NUM_STEPS,
                guidance_scale=CFG_SCALE,
                controlnet_conditioning_scale=sc_arg,
                generator=generator,
            )
            candidate = out.images[0]

            if best_of == 1:
                return candidate, None

            lp = compute_lpips(candidate, ref)
            if lp < best_lp:
                best_lp, best_img = lp, candidate

    return best_img, best_lp


print('✓ Img2Img pipeline loader and inference function defined')

## Step 7: Discover Validation Pairs

In [ ]:
EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

def load_val_pairs(val_dir: str) -> list:
    pairs = []
    for folder in sorted(os.listdir(val_dir), key=lambda x: int(x) if x.isdigit() else 9999):
        fp = os.path.join(val_dir, folder)
        if not os.path.isdir(fp):
            continue
        imgs = sorted([os.path.join(fp, f) for f in os.listdir(fp)
                       if os.path.splitext(f)[1].lower() in EXTS])
        if len(imgs) != 2:
            print(f'  ⚠  Folder {folder}: {len(imgs)} images (expected 2), skipping')
            continue
        pairs.append((imgs[0], imgs[1]))
    return pairs


val_pairs = load_val_pairs(VAL_DIR)
print(f'✓ {len(val_pairs)} pairs  →  {len(val_pairs)*2} eval samples per model')
for i, (a, b) in enumerate(val_pairs[:4]):
    print(f'  Pair {i+1}: {os.path.basename(a)}  ↔  {os.path.basename(b)}')

## Step 8: Pre-Cache ALL Control Maps (once, shared across 7 models)

In [ ]:
"""
Pre-compute every control map for every image in the validation set.
Stored in a dict keyed by (img_path, control_type).
This is the biggest A100 time-saver: control maps are computed ONCE
and reused across all 7 models instead of recomputed 7 × 42 times.
"""

# Collect unique image paths
all_img_paths = list({p for pair in val_pairs for p in pair})
all_images    = {p: Image.open(p).convert('RGB').resize((RESOLUTION, RESOLUTION))
                 for p in tqdm(all_img_paths, desc='Loading images')}

ctrl_cache = {}   # (path, control_type) → PIL Image

# ── Canny (CPU-bound, parallelise with threads) ────────────────────────────
print('Computing canny maps in parallel...')
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(make_canny, all_images[p]): p for p in all_img_paths}
    for fut, path in tqdm(futures.items(), total=len(futures), desc='Canny'):
        ctrl_cache[(path, 'canny')] = fut.result()

# ── Depth (GPU-bound, sequential but fast) ─────────────────────────────────
print('Computing depth maps...')
for path in tqdm(all_img_paths, desc='Depth'):
    ctrl_cache[(path, 'depth')] = make_depth(all_images[path])
    torch.cuda.empty_cache()

# ── Segmentation (GPU-bound, sequential) ──────────────────────────────────
print('Computing segmentation maps...')
for path in tqdm(all_img_paths, desc='Seg'):
    ctrl_cache[(path, 'seg')] = make_seg(all_images[path])
    torch.cuda.empty_cache()

torch.cuda.empty_cache(); gc.collect()
print(f'\n✓ Control map cache complete — {len(ctrl_cache)} entries cached')
print(f'  ({len(all_img_paths)} images × 3 control types)')

## Step 9: Visual Grid Helper

In [ ]:
CTRL_LABELS = {'canny': 'Canny', 'depth': 'Depth Map', 'seg': 'Seg Map'}
CTRL_CMAPS  = {'canny': 'gray', 'depth': 'inferno', 'seg': 'nipy_spectral'}


def show_best_grid(
    samples: list,
    config_name: str,
    controls: list,
    n_show: int = N_BEST_VIZ,
    best_of: int = 1,
    save_path: str = None,
):
    """
    Display a grid of the N best-performing samples for one model.

    Each row = one sample:
      Source | ControlMap1 [| ControlMap2 | ControlMap3] | Generated | Reference

    Samples are sorted by LPIPS (lowest = best = top rows).
    The best sample gets a green border; worst in the top-N gets an orange border.

    Args:
        samples : list of dicts from evaluate_model() containing images + scores
        config_name : e.g. 'Canny+Depth'
        controls    : list of control types for this config
        n_show      : number of rows to display
        best_of     : 1 or 4 (shown in title)
    """
    ranked = sorted(samples, key=lambda s: s['lpips'])[:n_show]
    n_ctrl = len(controls)
    n_cols  = 2 + n_ctrl      # source + controls + generated + reference
    col_labels = ['Source'] + [CTRL_LABELS[c] for c in controls] + ['Generated', 'Reference']

    fig_w = n_cols * 2.8
    fig_h = n_show * 2.8 + 1.0
    fig   = plt.figure(figsize=(fig_w, fig_h))
    fig.suptitle(
        f'Best {n_show} samples — {config_name}  |  '
        f'best-of-{best_of}  |  n=42',
        fontsize=14, fontweight='bold', y=1.01
    )

    gs = gridspec.GridSpec(n_show, n_cols, figure=fig,
                           hspace=0.35, wspace=0.08)

    for row_i, sample in enumerate(ranked):
        imgs = (
            [sample['source']]
            + [sample['ctrl_imgs'][c] for c in controls]
            + [sample['generated'], sample['reference']]
        )
        cmaps = (
            ['viridis' if False else None]        # source: RGB
            + [CTRL_CMAPS[c] for c in controls]
            + [None, None]                         # generated + reference: RGB
        )

        for col_i, (img, cmap) in enumerate(zip(imgs, cmaps)):
            ax = fig.add_subplot(gs[row_i, col_i])
            ax.imshow(img, cmap=cmap if cmap else None)
            ax.axis('off')

            # Column headers only on first row
            if row_i == 0:
                ax.set_title(col_labels[col_i], fontsize=9, fontweight='bold', pad=4)

            # Annotate generated image with scores
            is_gen_col = (col_i == 1 + n_ctrl)
            if is_gen_col:
                score_str = (
                    f"LPIPS {sample['lpips']:.4f}\n"
                    f"SSIM  {sample['ssim']:.4f}\n"
                    f"PSNR  {sample['psnr']:.2f} dB"
                )
                ax.text(4, RESOLUTION - 6, score_str,
                        fontsize=7, color='white', va='bottom',
                        bbox=dict(boxstyle='round,pad=0.3', fc='black', alpha=0.65))

                # Green border = best, orange = rest
                border_color = '#27ae60' if row_i == 0 else '#e67e22' if row_i == n_show-1 else '#95a5a6'
                for spine in ax.spines.values():
                    spine.set_edgecolor(border_color)
                    spine.set_linewidth(3)
                    spine.set_visible(True)

            # Row label: rank + condition
            if col_i == 0:
                label = f"#{row_i+1}"
                ax.set_ylabel(label, fontsize=9, rotation=0,
                              labelpad=18, va='center')

    if save_path:
        fig.savefig(save_path, dpi=130, bbox_inches='tight')
        print(f'  Saved: {save_path}')
    plt.show()
    plt.close(fig)


print('✓ show_best_grid() defined')

## Step 10: Core Evaluation Function

In [ ]:
def evaluate_model(
    config: dict,
    val_pairs: list,
    best_of: int = 1,
    visualize: bool = True,
) -> tuple:
    print(f"\n{'='*72}")
    print(f"  Config : {config['name']}  |  Controls: {', '.join(config['controls'])}  |  best_of={best_of}")
    print(f"{'='*72}")

    pipe    = load_pipeline(config)
    samples = []

    for pair_idx, (path_a, path_b) in enumerate(tqdm(val_pairs, desc=config['name'])):
        img_a = all_images[path_a]
        img_b = all_images[path_b]

        # Filename parse → CLIP fallback for each image
        cond_a = get_condition(path_a, img_a)
        cond_b = get_condition(path_b, img_b)

        for (src, ref, src_path, src_cond, tgt_cond) in [
            (img_a, img_b, path_a, cond_a, cond_b),   # A→B
            (img_b, img_a, path_b, cond_b, cond_a),   # B→A
        ]:
            cached_ctrls = [ctrl_cache[(src_path, c)] for c in config['controls']]
            building     = extract_building(src_path)
            pos, neg     = build_prompt(src_cond, tgt_cond, building)
            strength     = adaptive_strength(src_cond, tgt_cond)

            gen, lp_best_of = run_inference(
                pipe, src, cached_ctrls, config['controls'],
                pos, neg, strength,
                best_of=best_of, ref=ref,
            )

            lp  = lp_best_of if lp_best_of is not None else compute_lpips(gen, ref)
            ss  = compute_ssim(gen, ref)
            ps  = compute_psnr(gen, ref)

            gen_cond    = detect_condition_batch([gen])[0]
            time_acc    = int(gen_cond['time']    == tgt_cond['time'])
            weather_acc = int(gen_cond['weather'] == tgt_cond['weather'])

            samples.append({
                'pair':        pair_idx + 1,
                'source':      src,
                'reference':   ref,
                'generated':   gen,
                'ctrl_imgs':   {c: ctrl_cache[(src_path, c)] for c in config['controls']},
                'pos_prompt':  pos,
                'strength':    strength,
                'lpips':       lp,
                'ssim':        ss,
                'psnr':        ps,
                'time_acc':    time_acc,
                'weather_acc': weather_acc,
            })

    del pipe
    torch.cuda.empty_cache(); gc.collect()

    df = pd.DataFrame([{k: v for k, v in s.items()
                        if k not in ('source','reference','generated','ctrl_imgs','pos_prompt')}
                       for s in samples])

    means = {
        'config':      config['name'],
        'n':           len(df),
        'lpips':       df['lpips'].mean(),
        'ssim':        df['ssim'].mean(),
        'psnr':        df['psnr'].mean(),
        'time_acc':    df['time_acc'].mean() * 100,
        'weather_acc': df['weather_acc'].mean() * 100,
    }

    tag = f"{config['name']}_bo{best_of}"
    df.to_csv(f'{RESULTS_DIR}/{tag}_per_sample.csv', index=False)

    print(f"  LPIPS={means['lpips']:.4f}  SSIM={means['ssim']:.4f}  "
          f"PSNR={means['psnr']:.2f}  "
          f"TimeAcc={means['time_acc']:.1f}%  WeatherAcc={means['weather_acc']:.1f}%")

    # Print a few sample prompts so you can sanity-check them
    print(f"\n  Sample prompt (pair 1, A→B):\n    {samples[0]['pos_prompt'][:120]}...")
    print(f"  Adaptive strength used: {samples[0]['strength']}")

    if visualize:
        save_path = f"{RESULTS_DIR}/visuals/{tag}_best{N_BEST_VIZ}.png"
        show_best_grid(
            samples, config['name'], config['controls'],
            n_show=N_BEST_VIZ, best_of=best_of, save_path=save_path,
        )

    return means, samples


print('✓ evaluate_model() defined')

## Step 11: Run All 7 Models — Table 1 (best-of-1)
*(~5–8 min per config on A100 after first compile warm-up)*

In [ ]:
table1_rows, all_samples_bo1 = [], {}

for cfg_id in range(1, 8):
    means, samps = evaluate_model(CONFIGS[cfg_id], val_pairs, best_of=BEST_OF_1, visualize=True)
    table1_rows.append(means)
    all_samples_bo1[CONFIGS[cfg_id]['name']] = samps

table1 = pd.DataFrame(table1_rows)
table1.to_csv(f'{RESULTS_DIR}/table1_bo1.csv', index=False)
print('\n✓ Table 1 complete and saved')

## Step 12: Run All 7 Models — Table 2 (best-of-4)
*(~20–30 min per config — 4× inference + LPIPS selection per sample)*

In [ ]:
table2_rows, all_samples_bo4 = [], {}

for cfg_id in range(1, 8):
    means, samps = evaluate_model(CONFIGS[cfg_id], val_pairs, best_of=BEST_OF_4, visualize=True)
    table2_rows.append(means)
    all_samples_bo4[CONFIGS[cfg_id]['name']] = samps

table2 = pd.DataFrame(table2_rows)
table2.to_csv(f'{RESULTS_DIR}/table2_bo4.csv', index=False)
print('\n✓ Table 2 complete and saved')

## Step 13: Print Results Tables + LPIPS Bar Chart

In [ ]:
def render_results(df: pd.DataFrame, title: str, highlight_best: bool = True):
    disp = df[['config','n','lpips','ssim','psnr','time_acc','weather_acc']].copy()
    disp.columns = ['System','n','LPIPS ↓','SSIM ↑','PSNR ↑','Time Acc. ↑','Weather Acc. ↑']
    disp['LPIPS ↓']       = disp['LPIPS ↓'].map('{:.4f}'.format)
    disp['SSIM ↑']        = disp['SSIM ↑'].map('{:.4f}'.format)
    disp['PSNR ↑']        = disp['PSNR ↑'].map('{:.2f}'.format)
    disp['Time Acc. ↑']   = disp['Time Acc. ↑'].map('{:.1f}%'.format)
    disp['Weather Acc. ↑']= disp['Weather Acc. ↑'].map('{:.1f}%'.format)

    fig, ax = plt.subplots(figsize=(16, 0.55 + 0.52 * len(disp)))
    ax.axis('off')
    tbl = ax.table(cellText=disp.values, colLabels=disp.columns,
                   cellLoc='center', loc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1, 1.7)

    # Header row styling
    for j in range(len(disp.columns)):
        tbl[(0, j)].set_facecolor('#2c3e50')
        tbl[(0, j)].get_text().set_color('white')
        tbl[(0, j)].get_text().set_fontweight('bold')

    if highlight_best:
        best = int(df['lpips'].argmin())
        for j in range(len(disp.columns)):
            tbl[(best+1, j)].set_facecolor('#d5f5e3')
            tbl[(best+1, j)].get_text().set_fontweight('bold')

    plt.title(title, fontsize=13, fontweight='bold', pad=10, loc='left')
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/{title.replace(' ','_').replace('|','')[:60]}.png",
                dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(disp.to_string(index=False))


render_results(table1, 'Table 1 | Internal Val Ablation — best-of-1  (42 samples per config)')
render_results(table2, 'Table 2 | Course Staff Val    — best-of-4  (42 samples per config)')

## Step 14: LPIPS Ranking Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (df_data, label) in zip(axes, [
    (table1, 'Table 1 — best-of-1'),
    (table2, 'Table 2 — best-of-4'),
]):
    configs  = df_data['config'].tolist()
    lpips_v  = df_data['lpips'].tolist()
    best_idx = int(np.argmin(lpips_v))
    colors   = ['#27ae60' if i == best_idx else '#3498db' for i in range(len(configs))]

    bars = ax.bar(configs, lpips_v, color=colors, edgecolor='black', linewidth=0.8)
    ax.set_title(f'LPIPS per Config — {label}', fontweight='bold', fontsize=12)
    ax.set_ylabel('LPIPS ↓  (lower = better)')
    ax.set_ylim(0, max(lpips_v) * 1.22)
    ax.tick_params(axis='x', rotation=35)

    for bar, val in zip(bars, lpips_v):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8.5)

    ax.axhline(min(lpips_v), color='#27ae60', linestyle='--', linewidth=1, alpha=0.7)

plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/lpips_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

# Ranked text summary
print('\n LPIPS Ranking — Table 2 best-of-4 (lower = better)')
print(' ' + '-'*38)
for rank, (_, row) in enumerate(table2.sort_values('lpips').iterrows(), 1):
    star = ' ← best' if rank == 1 else ''
    print(f'  {rank}. {row["config"]:15s}  {row["lpips"]:.4f}{star}')

## Step 15: Cross-Model Best Sample Comparison

Shows the single best sample from EACH model side-by-side —
pick the same pair index so you can directly compare outputs.

In [ ]:
def cross_model_comparison(all_samples: dict, pair_idx: int = 0, direction: int = 0):
    """
    Show the generated output from every model for the same input.
    pair_idx : 0-based index into val_pairs
    direction: 0 = A→B, 1 = B→A
    """
    sample_i = pair_idx * 2 + direction   # index into each model's sample list

    config_names = list(all_samples.keys())
    n_models = len(config_names)

    fig, axes = plt.subplots(2, n_models + 2, figsize=((n_models + 2) * 2.8, 6.5))
    fig.suptitle(
        f'Cross-model comparison — Pair {pair_idx+1}, direction {direction}\n'
        f'Top row: Generated  |  Bottom row: Reference',
        fontsize=12, fontweight='bold'
    )

    ref_samp = all_samples[config_names[0]][sample_i]

    # Column 0: source
    axes[0, 0].imshow(ref_samp['source']); axes[0, 0].axis('off')
    axes[0, 0].set_title('Source', fontsize=9, fontweight='bold')
    axes[1, 0].imshow(ref_samp['reference']); axes[1, 0].axis('off')
    axes[1, 0].set_title('Reference\n(ground truth)', fontsize=9, fontweight='bold')

    lpips_vals = []
    for col, name in enumerate(config_names, start=1):
        s = all_samples[name][sample_i]
        lpips_vals.append(s['lpips'])
        axes[0, col].imshow(s['generated']); axes[0, col].axis('off')
        axes[0, col].set_title(f'{name}\nLPIPS {s["lpips"]:.4f}', fontsize=8.5)
        axes[1, col].axis('off')   # empty bottom

    # Highlight best generated
    best_col = int(np.argmin(lpips_vals)) + 1
    for spine in axes[0, best_col].spines.values():
        spine.set_edgecolor('#27ae60'); spine.set_linewidth(3); spine.set_visible(True)

    # Last column: reference again for easy visual comparison
    axes[0, -1].imshow(ref_samp['reference']); axes[0, -1].axis('off')
    axes[0, -1].set_title('Reference\n(repeated)', fontsize=8.5)
    axes[1, -1].axis('off')

    plt.tight_layout()
    save_p = f'{RESULTS_DIR}/visuals/cross_model_pair{pair_idx+1}_dir{direction}.png'
    fig.savefig(save_p, dpi=130, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Saved: {save_p}')


# Show cross-model comparison for the first 3 pairs
# (change pair_idx to inspect specific pairs)
for pi in range(min(3, len(val_pairs))):
    cross_model_comparison(all_samples_bo4, pair_idx=pi, direction=0)

---
## (Optional) Quick LPIPS-Only Pass

Run **only this cell** to get LPIPS numbers fast (skips SSIM/PSNR and condition accuracy).
Uses pre-cached control maps and the full A100 stack.
Uncomment the last line to execute.

In [ ]:
def quick_lpips(val_pairs: list) -> pd.DataFrame:
    """LPIPS-only pass across all 7 configs — skips SSIM/PSNR/accuracy. Best-of-1."""
    rows = []
    for cfg_id in range(1, 8):
        cfg    = CONFIGS[cfg_id]
        pipe   = load_pipeline(cfg)
        scores = []

        for path_a, path_b in tqdm(val_pairs, desc=cfg['name'], leave=False):
            img_a = all_images[path_a]
            img_b = all_images[path_b]
            for (src, ref, src_path, src_cond, tgt_cond) in [
                (img_a, img_b, path_a, get_condition(path_a, img_a), get_condition(path_b, img_b)),
                (img_b, img_a, path_b, get_condition(path_b, img_b), get_condition(path_a, img_a)),
            ]:
                ctrl_imgs = [ctrl_cache[(src_path, c)] for c in cfg['controls']]
                pos, neg  = build_prompt(src_cond, tgt_cond, extract_building(src_path))
                strength  = adaptive_strength(src_cond, tgt_cond)
                gen, _    = run_inference(pipe, src, ctrl_imgs, cfg['controls'],
                                          pos, neg, strength, best_of=1)
                scores.append(compute_lpips(gen, ref))

        del pipe; torch.cuda.empty_cache(); gc.collect()
        mean_lp = float(np.mean(scores))
        print(f'  {cfg["name"]:15s}  LPIPS = {mean_lp:.4f}  (n={len(scores)})')
        rows.append({'config': cfg['name'], 'n': len(scores), 'lpips': mean_lp})

    df = pd.DataFrame(rows)
    df.to_csv(f'{RESULTS_DIR}/quick_lpips.csv', index=False)
    return df


# Uncomment to run (fastest way to get all 7 LPIPS numbers):
# quick_df = quick_lpips(val_pairs)